# Module 2.1 — Document Loaders in LangChain

LangChain provides loaders for virtually every document type. Each loader returns a list of `Document` objects with `.page_content` and `.metadata`.

| Loader | Source |
|---|---|
| `PyPDFLoader` | PDF files |
| `PDFMinerLoader` | PDFs (precise layout) |
| `PDFPlumberLoader` | PDFs with tables |
| `UnstructuredPDFLoader` | Complex PDFs |
| `WebBaseLoader` | Web pages |
| `RecursiveUrlLoader` | Entire websites |
| `CSVLoader` | CSV files |
| `JSONLoader` | JSON / JSONL |
| `TextLoader` | Plain text |


In [ ]:
# !pip install langchain langchain-community pypdf pdfminer.six pdfplumber
# !pip install unstructured beautifulsoup4 requests

# ── 1. PDF Loaders ────────────────────────────────────────────────────────────
from langchain_community.document_loaders import (
    PyPDFLoader,
    PDFMinerLoader,
    PDFPlumberLoader,
)

def load_and_inspect(loader_class, path, **kwargs):
    loader = loader_class(path, **kwargs)
    docs   = loader.load()
    print(f"  Loader : {loader_class.__name__}")
    print(f"  Pages  : {len(docs)}")
    print(f"  Sample : {docs[0].page_content[:200].strip()}\n")
    return docs

# Replace with your actual PDF path
PDF_PATH = "sample.pdf"   # <-- change me

# Uncomment to test:
# load_and_inspect(PyPDFLoader,      PDF_PATH)
# load_and_inspect(PDFMinerLoader,   PDF_PATH)
# load_and_inspect(PDFPlumberLoader, PDF_PATH)


In [ ]:
# ── 2. Web Loaders ────────────────────────────────────────────────────────────
from langchain_community.document_loaders import WebBaseLoader, RecursiveUrlLoader
from langchain_community.document_loaders.recursive_url_loader import RecursiveUrlLoader
import bs4

# Single page
url  = "https://python.langchain.com/docs/introduction/"
loader = WebBaseLoader(web_paths=[url],
                       bs_kwargs={"parse_only": bs4.SoupStrainer(class_=("theme-doc-markdown",))})
web_docs = loader.load()
print(f"WebBaseLoader — chars: {len(web_docs[0].page_content)}")
print(web_docs[0].page_content[:300])


In [ ]:
# ── 3. CSV Loader ─────────────────────────────────────────────────────────────
import csv, tempfile, os
from langchain_community.document_loaders import CSVLoader

# Create a sample CSV
sample_csv = """product,description,price
Laptop,High performance laptop for developers,999
Phone,Latest smartphone with AI features,699
Tablet,Lightweight tablet for productivity,399
"""
with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False) as f:
    f.write(sample_csv)
    csv_path = f.name

loader   = CSVLoader(file_path=csv_path, csv_args={"delimiter": ","})
csv_docs = loader.load()
print(f"CSV rows loaded: {len(csv_docs)}")
for d in csv_docs:
    print(" ", d.page_content[:80])

os.unlink(csv_path)


In [ ]:
# ── 4. JSON Loader ────────────────────────────────────────────────────────────
import json, tempfile
from langchain_community.document_loaders import JSONLoader

sample_json = [
    {"title": "RAG Overview", "content": "RAG combines retrieval with generation."},
    {"title": "Embeddings",   "content": "Embeddings represent text as dense vectors."},
]
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
    json.dump(sample_json, f)
    json_path = f.name

loader    = JSONLoader(file_path=json_path, jq_schema=".[].content", text_content=False)
json_docs = loader.load()
print(f"JSON docs loaded: {len(json_docs)}")
for d in json_docs:
    print(" ", d.page_content)

os.unlink(json_path)


In [ ]:
# ── 5. Text Loader ───────────────────────────────────────────────────────────
from langchain_community.document_loaders import TextLoader

sample_text = "This is a plain text document.\nIt can contain multiple lines.\nUseful for logs and notes."
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    f.write(sample_text)
    txt_path = f.name

loader    = TextLoader(txt_path)
text_docs = loader.load()
print(f"Text docs: {len(text_docs)}")
print(text_docs[0].page_content)

os.unlink(txt_path)


## Hands-on Exercise
Load documents from 5 different sources and inspect their metadata and content. Try loading your own PDF, a company webpage, a CSV of product data, a JSON FAQ list, and a plain text readme.